# 9. Operational capstone: Website Maintenance Agent

One bounded scheduled tick fetches an update, compares durable state, obtains a structured proposal, applies named guardrails, pauses for approval, writes a real local website file, verifies it, and records the run. The classroom target is local Markdown; direct public publishing is outside the core.

## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Run an operational cycle from public/cached update through state, proposal, named guardrails, approval, persistent website change, verification, and events.

Architecture reference: [Day 5 diagrams D19](../../diagrams/source/day_05.md).

### Expected observation

The clean update pauses before writing; rejection changes no file; approval creates a verified Markdown update; poisoned external instructions are blocked. Exact IDs, timing, and live wording will vary.

## Concept briefing

## Automation is a trigger, not intelligence

A scheduler can start a run every day, but scheduling alone is ordinary automation. The
agentic decision is whether new evidence warrants a change and which permitted action to
propose. Policy then decides whether the exact proposal may proceed.

The Website Maintenance Agent demonstrates a production-shaped cycle at classroom scale:
fetch a real or cached public update, compare it with durable processed-item state, create
a structured website proposal, apply guardrails, pause for approval, write a real local
file, verify the result and record events. The scheduler should call one bounded `check`
operation; it should not contain hidden business logic.

An optional LLM judge may score whether the proposed update is faithful to its source.
That judge belongs after deterministic checks and before approval or publication. It is
advisory because it can be inconsistent, biased toward fluent text or influenced by the
content it evaluates. File-path, schema, source, build and permission checks remain
authoritative application code.


In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

## 1. Configure a fresh classroom run

The cached source is repeatable. The optional live source reads public GitHub release data. Both produce the same `UpdateItem` contract.

In [ ]:
from mini_harness import (CachedJSONSource,GitHubReleaseSource,JSONStateStore,WebsiteGuardrails,
    WebsiteMaintenanceAgent,deterministic_proposer,OpenRouterWebsiteProposer,EventStore)
run_root=DAY/"data"/"website_classroom_run"
site_root=run_root/"site"
source=CachedJSONSource(DAY/"data"/"website_updates.json")
events=EventStore(run_root/"events.jsonl")
state=JSONStateStore(run_root/"state.json")
guardrails=WebsiteGuardrails(site_root,{"github.com"})
proposer=deterministic_proposer
print("Website target:",site_root)

## 2. Check once and inspect the exact proposal

No website file exists yet. Approval is a state transition over the exact saved proposal, not a conversational "yes".

In [ ]:
agent=WebsiteMaintenanceAgent(source,proposer,guardrails,state,events)
pending=agent.check_once()
print(pending.status,pending.proposal)
assert pending.status in {"pending_approval","no_change"}
print("Website exists before approval:",(site_root/"content"/"updates.md").exists())

## 3. Resolve deliberately

For the first run, leave `approved=False` and prove rejection has no side effect. Use a fresh run directory before repeating with approval.

In [ ]:
if pending.status=="pending_approval":
    approved=False  # change only after inspecting the proposal
    final=agent.resolve(pending.run_id,approved)
    print(final.status,final.message)
print("Website exists:",(site_root/"content"/"updates.md").exists())

## 4. Practical indirect prompt-injection challenge

The poisoned fixture mixes a real-looking update with instructions to reveal a key and invoke another tool. External content is evidence, not authority.

In [ ]:
poisoned=WebsiteMaintenanceAgent(
    CachedJSONSource(DAY/"data"/"poisoned_website_updates.json"),deterministic_proposer,
    guardrails,JSONStateStore(run_root/"poisoned_state.json"),events)
blocked=poisoned.check_once()
print(blocked.status,blocked.message)
assert blocked.status=="blocked"
assert not (site_root/"content"/"updates.md").exists()

## 5. Optional bounded live observations

Choose one live source and one live model call. If unavailable, use the cached source and instructor-captured trace.

```python
source = GitHubReleaseSource("modelcontextprotocol", "python-sdk", limit=3)
# proposer = OpenRouterWebsiteProposer()
```

Never publish automatically in this course. A live run stops at `pending_approval`.

## 6. Evaluation and optional LLM judge

Deterministic checks remain authoritative: trusted host, matching evidence, allowed path, body-size limit, prohibited active content, explicit approval and post-write verification. An optional LLM judge may score semantic faithfulness, but it is advisory and requires calibration against human-labelled examples.

A model council is unnecessary unless measured evidence shows one proposer/reviewer is inadequate. Day 4 provides the specialist-and-supervisor pattern.

## 7. Daily automation boundary

An operating-system scheduler, cron or CI schedule invokes `run_website_agent.py` once per day. Scheduling is ordinary automation; it merely triggers one bounded check. Production credentials, deployment and unattended approval are outside the core.

## Required live observation

Choose one bounded live observation: fetch up to three public releases or obtain one OpenRouter update proposal. Stop before approval. The cached source and captured trace are the outage fallback.


## Your turn

Run the cached cycle, reject once, approve once in a fresh state directory, and explain which controls remain authoritative with a live model.

## Recap

A scheduler triggers a bounded run; the agent proposes; guardrails and a human control the real side effect. Name one responsibility that deliberately remains application-specific.